# Banco Río: perfiles de cartera y riesgo de crédito

Proyecto final de Programación Analítica II (Universidad de Montevideo, S2 2026). Propuesta A.

Fecha de corte de los datos: 30/06/2026. Originación: enero 2023 a marzo 2026.

## 1. EDA y calidad de datos

### 1.1 Carga de los tres archivos declarando tipos y fechas

Criterios de la carga:

- **Identificadores como texto** (`id_cliente`, `id_producto`): son etiquetas, no cantidades; nunca se suman ni se promedian.
- **Columnas de texto como `str`**, no como `category`: varias vienen escritas de distintas maneras ("Montevideo" / "montevideo"). Si se convierten a categoría ahora, cada variante quedaría como una categoría distinta. Se convierten después de estandarizarlas.
- **`ingreso_mensual` como texto**: viene en tres formatos ("34500.00", "$ 55.400", "29.100,00"). Si se lee como número, pandas falla o descarta valores. Se convierte en el punto 1.6.
- **Enteros con faltantes como `float64`**: `int64` no admite `NaN`.
- **Fechas con el formato declarado**: así pandas no adivina y un formato inesperado queda a la vista. Dos columnas de productos mezclan `AAAA-MM-DD` y `DD/MM/AAAA` y se convierten aparte.

In [1]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

In [2]:
# Rutas: el notebook vive en notebooks/, los datos están una carpeta más arriba
CARPETA_DATOS = Path("..") / "datos"
RUTA_CLIENTES = CARPETA_DATOS / "clientes.csv"
RUTA_PRODUCTOS = CARPETA_DATOS / "productos.csv"
RUTA_TIPO_CAMBIO = CARPETA_DATOS / "tipo_cambio_mensual.csv"

FILAS_A_MOSTRAR = 5

# Formatos de fecha que aparecen en los archivos
FORMATO_ISO = "%Y-%m-%d"
FORMATO_DIA_MES_ANIO = "%d/%m/%Y"
FORMATO_MES = "%Y-%m"
PATRON_ISO = r"^\d{4}-\d{2}-\d{2}$"
PATRON_DIA_MES_ANIO = r"^\d{2}/\d{2}/\d{4}$"

TIPOS_CLIENTES = {
    "id_cliente": "str",
    "edad": "float64",  # tiene faltantes
    "sexo": "str",
    "estado_civil": "str",
    "cantidad_dependientes": "int64",
    "departamento": "str",
    "nivel_educativo": "str",
    "sector_empleo": "str",
    "antiguedad_laboral_meses": "float64",  # tiene faltantes
    "situacion_vivienda": "str",
    "ingreso_mensual": "str",  # tres formatos distintos, se convierte en 1.6
    "score_clearing": "float64",  # tiene faltantes
    "canal_originacion": "str",
    "segmento_comercial": "str",
    "tiene_cuenta_sueldo": "str",
}
FECHAS_CLIENTES = {"fecha_alta_cliente": FORMATO_ISO}

TIPOS_PRODUCTOS = {
    "id_producto": "str",
    "id_cliente": "str",  # texto, igual que en clientes, para que el merge coincida
    "tipo_producto": "str",
    "moneda": "str",
    "fecha_alta_producto": "str",  # dos formatos, se convierte aparte
    "plazo_meses": "int64",
    "tasa_nominal_anual": "float64",
    "score_originacion": "float64",  # tiene faltantes
    "monto_otorgado": "float64",
    "cuota_mensual": "float64",
    "limite_credito": "float64",
    "saldo_actual": "float64",
    "saldo_maximo_historico": "float64",
    "cuotas_pagas_en_termino": "int64",
    "veces_atraso_leve": "int64",
    "veces_atraso_medio": "int64",
    "veces_atraso_grave": "int64",
    "dias_atraso_acumulados": "int64",
    "fecha_primer_atraso_grave": "str",  # dos formatos, se convierte aparte
}
FECHAS_PRODUCTOS = {
    "fecha_ultima_actualizacion": FORMATO_ISO,
    "fecha_ultimo_reporte": FORMATO_DIA_MES_ANIO,
}
COLUMNAS_FECHA_DOS_FORMATOS = ["fecha_alta_producto", "fecha_primer_atraso_grave"]

TIPOS_TIPO_CAMBIO = {"tipo_cambio_usd": "float64", "valor_unidad_indexada": "float64"}
FECHAS_TIPO_CAMBIO = {"mes": FORMATO_MES}

# Categorías válidas de cada columna según el enunciado (los 19 departamentos de
# Uruguay para departamento). Las uso para contar y para estandarizar.
DEPARTAMENTOS_URUGUAY = [
    "Artigas", "Canelones", "Cerro Largo", "Colonia", "Durazno", "Flores",
    "Florida", "Lavalleja", "Maldonado", "Montevideo", "Paysandú", "Río Negro",
    "Rivera", "Rocha", "Salto", "San José", "Soriano", "Tacuarembó",
    "Treinta y Tres",
]  # fmt: skip
CATEGORIAS_VALIDAS = {
    "sexo": ["F", "M"],
    "estado_civil": ["Soltero/a", "Casado/a", "Unión libre", "Divorciado/a", "Viudo/a"],
    "departamento": DEPARTAMENTOS_URUGUAY,
    "sector_empleo": ["Público", "Privado", "Independiente", "Jubilado", "Informal"],
    "situacion_vivienda": ["Propia", "Propia con hipoteca", "Alquilada", "Familiar"],
    "canal_originacion": [
        "Sucursal", "Web", "App", "Comercio adherido", "Telemarketing",
    ],
    "segmento_comercial": ["Masivo", "Preferente", "Premium"],
    "tiene_cuenta_sueldo": ["Si", "No"],
    "tipo_producto": [
        "Tarjeta de crédito", "Préstamo al consumo", "Adelanto de sueldo",
        "Préstamo automotor", "Crédito hipotecario", "Línea de crédito",
    ],
    "moneda": ["UYU", "USD", "UI"],
}  # fmt: skip
CATEGORIAS_ESPERADAS = {
    columna: len(valores) for columna, valores in CATEGORIAS_VALIDAS.items()
}

# Ingreso: "$ 55.400" usa el punto como separador de miles (punto seguido de 3 dígitos)
PATRON_MILES_CON_PUNTO = r"^\d{1,3}(\.\d{3})+$"
# Los ingresos declarados vienen redondeados a la centena (lo verifico en el punto 1.3)
REDONDEO_INGRESO = 100

# Escrituras que no coinciden con la categoría válida ni cambiando mayúsculas,
# tildes y espacios. "Changas" es trabajo informal, por eso lo paso a Informal.
SINONIMOS_CATEGORIAS = {
    "departamento": {"San Jose de Mayo": "San José"},
    "sector_empleo": {
        "Jubilado/a": "Jubilado",
        "Independ.": "Independiente",
        "Sector publico": "Público",
        "Changas": "Informal",
    },
    "situacion_vivienda": {"Alquiler": "Alquilada", "Cedida por familiar": "Familiar"},
    "canal_originacion": {
        "Sitio web": "Web",
        "APP MOVIL": "App",
        "Call center": "Telemarketing",
    },
    "tipo_producto": {
        "Prestamo consumo": "Préstamo al consumo",
        "Adelanto sueldo": "Adelanto de sueldo",
    },
}

# Rangos que admite el negocio. Lo que cae afuera es un error o un código de "no sé".
SIN_LIMITE = np.inf
RANGOS_VALIDOS = {
    "edad": (18, 100),  # no se le presta a menores de 18
    "antiguedad_laboral_meses": (0, SIN_LIMITE),
    "score_clearing": (300, 950),  # escala del Clearing según el enunciado
    "score_originacion": (300, 950),
    "plazo_meses": (1, 360),  # hasta 30 años, el plazo de un hipotecario
    "monto_otorgado": (0, SIN_LIMITE),
    "dias_atraso_acumulados": (0, SIN_LIMITE),
}
EDAD_MINIMA_PARA_TRABAJAR = 14
MESES_POR_ANIO = 12

# Una tasa de consumo como fracción llega como mucho a 1,10 (110% anual); escrita
# en porcentaje arranca en 5,72. Todo lo que pasa de 1,5 lo tomo como porcentaje.
UMBRAL_TASA_EN_PORCENTAJE = 1.5
PORCENTAJE_A_FRACCION = 100

# Etapa 2: definición de mora y monedas
FECHA_CORTE = pd.Timestamp("2026-06-30")
# Ventana para comparar créditos con la misma oportunidad de atrasarse (punto 2.1)
MESES_VENTANA_MORA = 6
VENTANA_MORA = pd.DateOffset(months=MESES_VENTANA_MORA)
# Columna de tipo_cambio que pasa a pesos cada moneda (el peso no necesita cotización)
COTIZACION_POR_MONEDA = {"USD": "tipo_cambio_usd", "UI": "valor_unidad_indexada"}

In [3]:
def cargar_tabla(ruta, tipos_columnas, formatos_fecha):
    """Lee un CSV declarando el tipo de cada columna y el formato de cada fecha."""
    return pd.read_csv(
        ruta,
        dtype=tipos_columnas,
        parse_dates=list(formatos_fecha),
        date_format=formatos_fecha,
    )


def convertir_fecha_dos_formatos(serie_texto):
    """Convierte a fecha una columna que mezcla AAAA-MM-DD y DD/MM/AAAA."""
    fecha_iso = pd.to_datetime(serie_texto, format=FORMATO_ISO, errors="coerce")
    fecha_dia_mes_anio = pd.to_datetime(
        serie_texto, format=FORMATO_DIA_MES_ANIO, errors="coerce"
    )
    return fecha_iso.fillna(fecha_dia_mes_anio)


def resumir_conversion_fecha(serie_texto, serie_fecha):
    """Cuenta las filas de cada formato y las que no se pudieron convertir."""
    filas_con_dato = serie_texto.notna().sum()
    filas_dia_mes_anio = serie_texto.str.match(PATRON_DIA_MES_ANIO).sum()
    return {
        "filas_con_dato": filas_con_dato,
        "formato_iso": serie_texto.str.match(PATRON_ISO).sum(),
        "formato_dia_mes_anio": filas_dia_mes_anio,
        "pct_dia_mes_anio": round(100 * filas_dia_mes_anio / filas_con_dato, 1),
        "fechas_no_convertidas": (serie_texto.notna() & serie_fecha.isna()).sum(),
    }


def resumir_columnas_fecha(tabla):
    """Devuelve el rango y los faltantes de cada columna de fecha de la tabla."""
    columnas_fecha = tabla.select_dtypes(include="datetime").columns
    return pd.DataFrame(
        {
            "fecha_minima": tabla[columnas_fecha].min(),
            "fecha_maxima": tabla[columnas_fecha].max(),
            "faltantes": tabla[columnas_fecha].isna().sum(),
        }
    )

In [4]:
clientes = cargar_tabla(
    ruta=RUTA_CLIENTES,
    tipos_columnas=TIPOS_CLIENTES,
    formatos_fecha=FECHAS_CLIENTES,
)
productos = cargar_tabla(
    ruta=RUTA_PRODUCTOS,
    tipos_columnas=TIPOS_PRODUCTOS,
    formatos_fecha=FECHAS_PRODUCTOS,
)
tipo_cambio = cargar_tabla(
    ruta=RUTA_TIPO_CAMBIO,
    tipos_columnas=TIPOS_TIPO_CAMBIO,
    formatos_fecha=FECHAS_TIPO_CAMBIO,
)

**Fechas con dos formatos.** Se convierte cada columna probando los dos formatos y se reporta cuántas filas venían en cada uno. Si alguna fecha quedara sin convertir (`fechas_no_convertidas` > 0) habría un tercer formato que no vimos.

In [5]:
# Modificación: texto -> fecha. Se reporta el % de filas que venía en DD/MM/AAAA
# y se verifica que ninguna fecha con dato se pierda en la conversión.
resumen_conversion = {}
for columna_fecha in COLUMNAS_FECHA_DOS_FORMATOS:
    fecha_convertida = convertir_fecha_dos_formatos(productos[columna_fecha])
    resumen_conversion[columna_fecha] = resumir_conversion_fecha(
        productos[columna_fecha], fecha_convertida
    )
    productos[columna_fecha] = fecha_convertida

display(pd.DataFrame.from_dict(resumen_conversion, orient="index"))

,filas_con_dato,formato_iso,formato_dia_mes_anio,pct_dia_mes_anio,fechas_no_convertidas
fecha_alta_producto,6191,5321,870,14.1,0
fecha_primer_atraso_grave,2438,2124,314,12.9,0


**Qué encontramos:** el 14,1% de las fechas de alta de productos (870 de 6.191) y el 12,9% de las fechas de primer atraso grave (314 de 2.438) venían en formato `DD/MM/AAAA`; el resto en `AAAA-MM-DD`.

**Evidencia:** las 8.629 fechas con dato se convirtieron y ninguna quedó sin convertir (`fechas_no_convertidas` = 0), así que no hay un tercer formato escondido. Los faltantes de `fecha_primer_atraso_grave` ya venían vacíos en el archivo: no los generó la conversión.

**Qué sugiere:** si se hubiera dejado que pandas adivinara el formato, una fecha como 03/04/2025 podía leerse como 4 de marzo en lugar de 3 de abril y mover créditos de trimestre, que es justamente el corte que usamos para ver si la cartera mejora o empeora.

**Primeras filas de cada tabla.**

In [6]:
display(clientes.head(FILAS_A_MOSTRAR))
display(productos.head(FILAS_A_MOSTRAR))
display(tipo_cambio.head(FILAS_A_MOSTRAR))

,id_cliente,fecha_alta_cliente,edad,sexo,estado_civil,cantidad_dependientes,departamento,nivel_educativo,sector_empleo,antiguedad_laboral_meses,situacion_vivienda,ingreso_mensual,score_clearing,canal_originacion,segmento_comercial,tiene_cuenta_sueldo
0,7807176,2021-10-09,70.0,F,Soltero/a,1,Montevideo,Secundaria completa,Jubilado,0.0,Propia con hipoteca,34500.00,603.0,Sucursal,Masivo,No
1,3180655,2022-07-11,52.0,M,Soltero/a,1,Montevideo,Secundaria completa,Público,51.0,Propia,$ 55.400,597.0,Comercio adherido,Masivo,No
2,3028040,2020-11-08,42.0,F,Casado/a,0,Maldonado,Secundaria completa,Independiente,6.0,Alquiler,100700,546.0,Sucursal,Preferente,No
3,6712544,2018-08-02,59.0,F,Soltero/a,1,Tacuarembó,Secundaria completa,Privado,42.0,Alquiler,90500.00,619.0,Sucursal,Preferente,No
4,8244977,2022-06-27,47.0,F,Unión libre,3,Canelones,Terciaria completa,Privado,50.0,Propia,"59.500,00",689.0,Web,Masivo,Si


,id_producto,id_cliente,tipo_producto,moneda,fecha_alta_producto,plazo_meses,tasa_nominal_anual,score_originacion,monto_otorgado,cuota_mensual,...,saldo_actual,saldo_maximo_historico,cuotas_pagas_en_termino,veces_atraso_leve,veces_atraso_medio,veces_atraso_grave,dias_atraso_acumulados,fecha_primer_atraso_grave,fecha_ultima_actualizacion,fecha_ultimo_reporte
0,900023,10817904,Adelanto de sueldo,UYU,2023-03-22,10,0.7533,693.0,42924.53,5908.98,...,0.00,NaN,10,0,0,0,0,NaT,2026-06-19,2026-05-22
1,900035,10504807,Tarjeta de crédito,UYU,2024-08-27,12,0.9147,601.0,250835.46,32636.20,...,0.00,304682.81,6,3,2,1,121,2025-05-19,2026-06-17,NaT
2,505812,1113415,Préstamo automotor,USD,2024-02-24,31,0.1052,690.0,24043.55,889.12,...,2262.21,27384.35,19,4,2,1,213,2025-11-10,2026-06-13,2026-04-01
3,500952,4857398,Préstamo automotor,USD,2024-10-26,43,0.1493,610.0,22294.55,672.58,...,13258.66,24836.47,16,2,2,0,104,NaT,2026-06-08,2026-03-21
4,505792,6176407,Adelanto de sueldo,UYU,2023-03-21,6,1.0948,722.0,80960.83,18114.34,...,0.00,81084.13,5,1,0,0,19,NaT,2026-06-06,2026-04-10


,mes,tipo_cambio_usd,valor_unidad_indexada
0,2023-01-01,39.037,5.6589
1,2023-02-01,38.729,5.6993
2,2023-03-01,39.448,5.7434
3,2023-04-01,40.052,5.7729
4,2023-05-01,40.124,5.7937


**Verificación de la carga.** Se controla que cada columna tenga el tipo declarado y que las fechas caigan en rangos razonables: altas de productos entre enero 2023 y marzo 2026, nada posterior al corte del 30/06/2026 y tipo de cambio para todos los meses de otorgamiento.

In [7]:
tipos_cargados = pd.concat(
    {
        "clientes": clientes.dtypes.astype(str),
        "productos": productos.dtypes.astype(str),
        "tipo_cambio": tipo_cambio.dtypes.astype(str),
    }
)
display(tipos_cargados.value_counts().to_frame("cantidad_columnas"))

,cantidad_columnas
str,15
float64,12
int64,7
datetime64[us],6


In [8]:
display(resumir_columnas_fecha(clientes))
display(resumir_columnas_fecha(productos))
display(resumir_columnas_fecha(tipo_cambio))

,fecha_minima,fecha_maxima,faltantes
fecha_alta_cliente,2013-02-24,2022-12-02,0


,fecha_minima,fecha_maxima,faltantes
fecha_alta_producto,2023-01-01,2026-03-30,0
fecha_primer_atraso_grave,2023-03-28,2026-06-24,3753
fecha_ultima_actualizacion,2026-05-17,2026-06-30,0
fecha_ultimo_reporte,2026-02-17,2026-06-30,567


,fecha_minima,fecha_maxima,faltantes
mes,2023-01-01,2026-06-01,0


**Qué encontramos:** las 40 columnas quedaron con el tipo declarado (16 de clientes + 21 de productos + 3 de tipo de cambio = 15 de texto, 12 decimales, 7 enteras y 6 fechas) y las fechas caen donde deben: las altas de productos van del 01/01/2023 al 30/03/2026, igual que el período de originación del enunciado, y ninguna fecha pasa del corte del 30/06/2026.

**Evidencia:** los clientes se dieron de alta entre 2013 y diciembre de 2022, antes de su primer producto; el tipo de cambio cubre de enero 2023 a junio 2026 (42 meses), así que todos los meses de otorgamiento tienen cotización para pasar los montos a pesos.

**Qué sugiere y qué falta:** la carga es confiable para seguir. Quedan dos grupos de vacíos para el punto 1.4. Los 3.753 de `fecha_primer_atraso_grave` no son datos perdidos: según el enunciado la fecha queda "vacía si nunca pasó", y eso hay que confirmarlo cruzándola con `veces_atraso_grave`. Los 567 de `fecha_ultimo_reporte` no tienen explicación en el enunciado: hay que ver si se concentran en algún grupo.

### 1.2 Resumen de cada tabla

Antes de limpiar nada se saca una "foto" de cada tabla: cuántas filas y columnas tiene, cuántas categorías distintas hay en cada columna de texto (comparadas con las que debería tener según el enunciado) y la descripción estadística de las columnas numéricas. Los tipos de cada columna ya se verificaron en el punto 1.1.

Esta foto no corrige nada: sirve para ver dónde están los problemas y medir después cuánto cambió cada cosa al limpiar.

In [9]:
def resumir_dimensiones(tablas):
    """Devuelve la cantidad de filas y columnas de cada tabla."""
    return pd.DataFrame(
        {
            nombre: {"filas": tabla.shape[0], "columnas": tabla.shape[1]}
            for nombre, tabla in tablas.items()
        }
    ).T


def contar_categorias(tabla, categorias_esperadas):
    """Cuenta los valores distintos de cada columna de texto contra los esperados."""
    columnas_texto = tabla.select_dtypes(include="str").columns
    resumen = tabla[columnas_texto].nunique().to_frame("categorias_en_archivo")
    resumen["categorias_esperadas"] = (
        resumen.index.to_series().map(categorias_esperadas).astype("Int64")
    )
    return resumen.sort_values("categorias_en_archivo", ascending=False)


def describir_continuas(tabla):
    """Devuelve la descripción estadística de las columnas numéricas."""
    return tabla.select_dtypes(include="number").describe().T.round(2)

In [10]:
# Junto las tres tablas en un diccionario para resumirlas con la misma función
tablas_del_banco = {
    "clientes": clientes,
    "productos": productos,
    "tipo_cambio": tipo_cambio,
}
display(resumir_dimensiones(tablas_del_banco))

,filas,columnas
clientes,3235,16
productos,6191,21
tipo_cambio,42,3


**Qué encontramos:** clientes tiene 3.235 filas y 16 columnas, productos 6.191 filas y 21 columnas, y tipo de cambio 42 filas y 3 columnas. Según el enunciado, una fila de clientes es un cliente, una fila de productos es un producto de crédito (un cliente puede tener varios) y una fila de tipo de cambio es un mes (de enero 2023 a junio 2026).

**Evidencia:** en clientes debería haber una fila por cliente, pero hay 3.235 filas y solo 3.200 `id_cliente` distintos (tabla de abajo); en productos hay 6.191 filas y 6.051 `id_producto` distintos. Sobran 35 filas en clientes y 140 en productos.

**Qué sugiere:** hay filas repetidas en las dos tablas. Si no se resuelven antes del merge, un cliente repetido duplicaría sus productos y la mora quedaría mal calculada. Se separan en el punto 1.3 (duplicados exactos contra id repetido con datos distintos).

#### Clientes

In [11]:
display(contar_categorias(clientes, CATEGORIAS_ESPERADAS))
display(describir_continuas(clientes))

,categorias_en_archivo,categorias_esperadas
id_cliente,3200,<NA>
ingreso_mensual,2083,<NA>
departamento,55,19
canal_originacion,10,5
sector_empleo,10,5
situacion_vivienda,7,4
nivel_educativo,6,<NA>
estado_civil,5,5
segmento_comercial,3,3
sexo,2,2


,count,mean,std,min,25%,50%,75%,max
edad,3225.0,55.52,15.57,0.0,46.0,54.0,63.0,202.0
cantidad_dependientes,3235.0,1.11,1.06,0.0,0.0,1.0,2.0,6.0
antiguedad_laboral_meses,3018.0,55.19,74.27,-33.0,9.0,37.0,80.0,771.0
score_clearing,3094.0,647.62,125.61,0.0,595.0,659.0,719.0,950.0


**Qué encontramos:** 4 columnas de texto tienen más categorías que las que deberían: `departamento` tiene 55 cuando Uruguay tiene 19, `canal_originacion` y `sector_empleo` tienen 10 cuando deberían ser 5, y `situacion_vivienda` tiene 7 cuando deberían ser 4. En las numéricas hay valores imposibles: edad mínima 0 y máxima 202, antigüedad laboral mínima de -33 meses y `score_clearing` mínimo de 0 cuando la escala va de 300 a 950.

**Evidencia:** conteo de valores distintos sobre las 3.235 filas y `describe` de las 4 columnas numéricas. `ingreso_mensual` no aparece en el `describe` porque se leyó como texto (sus 2.083 valores distintos vienen en tres formatos).

**Qué sugiere:** las categorías de más son la misma categoría escrita de varias maneras ("Montevideo" / "montevideo"), y se estandarizan en el punto 1.5 junto con los valores imposibles. Falta saber si esos valores imposibles son errores de carga o códigos que el sistema usa para decir "no sé".

#### Productos

In [12]:
display(contar_categorias(productos, CATEGORIAS_ESPERADAS))
display(describir_continuas(productos))

,categorias_en_archivo,categorias_esperadas
id_producto,6051,<NA>
id_cliente,3296,<NA>
tipo_producto,10,6
moneda,3,3


,count,mean,std,min,25%,50%,75%,max
plazo_meses,6191.0,34.48,65.25,0.00,12.00,13.00,37.00,999.00
tasa_nominal_anual,6191.0,4.42,16.28,0.06,0.53,0.71,0.87,109.34
score_originacion,6035.0,637.59,115.81,0.00,580.00,648.00,708.00,950.00
monto_otorgado,6191.0,219583.18,800567.16,-460190.39,54919.57,121710.11,235695.05,33887347.00
cuota_mensual,6191.0,15988.57,18569.72,43.09,4938.16,10525.49,20772.67,390273.93
limite_credito,5351.0,273129.92,328559.06,2279.95,77868.98,173533.56,339680.15,4864991.70
saldo_actual,6191.0,67827.66,170196.52,0.00,0.00,0.00,55920.67,1759940.54
saldo_maximo_historico,5892.0,212979.95,253229.95,1855.52,61751.19,137521.40,263475.91,3894419.72
cuotas_pagas_en_termino,6191.0,11.55,8.16,0.00,6.00,10.00,15.00,41.00
veces_atraso_leve,6191.0,1.37,1.38,0.00,0.00,1.00,2.00,12.00


**Qué encontramos:** `tipo_producto` tiene 10 categorías cuando el enunciado lista 6. En las numéricas hay cuatro señales de problemas: `plazo_meses` llega a 999 y tiene mínimo 0; `tasa_nominal_anual` tiene promedio 4,42 pero mediana 0,71; `monto_otorgado` tiene montos negativos (mínimo -460.190); y `dias_atraso_acumulados` tiene mínimo -999.

**Evidencia:** conteo de valores distintos y `describe` de las 13 columnas numéricas sobre las 6.191 filas. En la tasa, el promedio es 6 veces la mediana: la mayoría de las tasas están entre 0,53 y 0,87 (fracción, o sea 53% a 87%), pero unas pocas llegan a 109,34, que solo tiene sentido si están escritas en porcentaje.

**Qué sugiere:** la tasa es la candidata a columna con dos unidades (punto 1.6), y el 999 y el -999 parecen códigos de "no sé" (punto 1.5). Además, `monto_otorgado` y `cuota_mensual` mezclan pesos, dólares y UI, así que su promedio todavía no significa nada: hay que pasarlos a pesos con el tipo de cambio del mes de otorgamiento (punto 2.2). Otra cosa a mirar: productos tiene 3.296 `id_cliente` distintos y clientes solo 3.200, así que hay productos de clientes que no están en la tabla de clientes.

#### Tipo de cambio

In [13]:
display(describir_continuas(tipo_cambio))

,count,mean,std,min,25%,50%,75%,max
tipo_cambio_usd,42.0,42.04,2.08,38.73,40.72,41.33,44.40,45.52
valor_unidad_indexada,42.0,6.19,0.32,5.66,5.89,6.19,6.47,6.75


**Qué encontramos:** el dólar se movió entre $38,73 y $45,52 y la unidad indexada entre $5,66 y $6,75 en los 42 meses, sin faltantes ni valores raros.

**Evidencia:** `describe` de las 2 columnas sobre las 42 filas. La tabla no tiene columnas de texto, por eso no lleva conteo de categorías.

**Qué sugiere:** la tabla está limpia y se puede usar tal cual para convertir montos. Entre el mínimo y el máximo del dólar hay más de 17% de diferencia, así que usar un tipo de cambio fijo en lugar del del mes de otorgamiento movería bastante los montos en pesos.

### 1.3 Duplicados exactos contra identificador repetido con datos distintos

Son dos problemas distintos y se resuelven distinto:

- **Duplicado exacto:** la misma fila aparece dos veces, idéntica en todas las columnas. Es una doble carga: no aporta información y se elimina sin discusión.
- **Identificador repetido con datos distintos:** el mismo `id_cliente` (o `id_producto`) aparece dos veces, pero con algún dato diferente. No alcanza con borrar "la segunda": hay que mirar **qué columna cambia** y decidir con un criterio escrito cuál de las dos versiones es la buena.

Por la regla de la etapa, cada eliminación se anota en una bitácora con las filas antes y después y el porcentaje de la base afectado.

In [14]:
def contar_duplicados(tabla, columna_id):
    """Cuenta las filas idénticas y los identificadores que aparecen más de una vez."""
    sin_exactos = tabla.drop_duplicates()
    return {
        "filas": len(tabla),
        "duplicados_exactos": tabla.duplicated().sum(),
        "ids_repetidos_en_total": tabla[columna_id].duplicated().sum(),
        "ids_repetidos_con_datos_distintos": sin_exactos[columna_id].duplicated().sum(),
    }


def registrar_cambio(tabla_afectada, descripcion, filas_antes, filas_despues):
    """Devuelve una fila de la bitácora con el porcentaje de la base que se eliminó."""
    filas_eliminadas = filas_antes - filas_despues
    return {
        "tabla": tabla_afectada,
        "cambio": descripcion,
        "filas_antes": filas_antes,
        "filas_despues": filas_despues,
        "filas_eliminadas": filas_eliminadas,
        "pct_eliminado": round(100 * filas_eliminadas / filas_antes, 2),
    }


def contar_columnas_que_difieren(tabla, columna_id):
    """Cuenta, para los ids repetidos, en cuántos de ellos cambia cada columna."""
    repetidos = tabla[tabla[columna_id].duplicated(keep=False)]
    cambia_por_id = repetidos.groupby(columna_id).nunique(dropna=False) > 1
    cantidad_por_columna = cambia_por_id.sum().sort_values(ascending=False)
    return cantidad_por_columna[cantidad_por_columna > 0].to_frame("ids_donde_cambia")


def convertir_ingreso_a_numero(ingreso_texto):
    """Convierte a número un ingreso escrito en cualquiera de los formatos del archivo.

    Si hay coma, la coma es decimal y el punto separa miles ("29.100,00"). Si no hay
    coma, un punto seguido de 3 dígitos también separa miles ("$ 55.400").
    """
    if pd.isna(ingreso_texto):
        return np.nan
    texto_limpio = ingreso_texto.replace("$", "").replace(" ", "")
    if "," in texto_limpio:
        texto_limpio = texto_limpio.replace(".", "").replace(",", ".")
    elif re.match(PATRON_MILES_CON_PUNTO, texto_limpio):
        texto_limpio = texto_limpio.replace(".", "")
    try:
        return float(texto_limpio)
    except ValueError:
        # Textos como "S/D" o "SIN DATO": no hay número, los trato en el punto 1.5
        return np.nan

In [15]:
display(
    pd.DataFrame(
        {
            "clientes": contar_duplicados(clientes, "id_cliente"),
            "productos": contar_duplicados(productos, "id_producto"),
        }
    ).T
)

,filas,duplicados_exactos,ids_repetidos_en_total,ids_repetidos_con_datos_distintos
clientes,3235,24,35,11
productos,6191,108,140,32


**Qué encontramos:** en clientes hay 24 filas exactamente duplicadas y 11 `id_cliente` que se repiten con datos distintos (35 repeticiones en total, las que vimos en el punto 1.2). En productos hay 108 duplicados exactos y 32 `id_producto` repetidos con datos distintos (140 en total).

**Evidencia:** `duplicated()` sobre todas las columnas para los exactos, y `duplicated()` sobre el identificador después de sacar los exactos para los parciales. Los dos casos suman lo que sobraba en el 1.2: 24 + 11 = 35 en clientes y 108 + 32 = 140 en productos.

**Qué sugiere:** los exactos se eliminan ya. Los parciales hay que mirarlos uno por uno antes de decidir.

In [16]:
# Elimino los duplicados exactos: son la misma fila cargada dos veces
bitacora_limpieza = []

filas_antes_del_cambio = len(clientes)
clientes = clientes.drop_duplicates()
bitacora_limpieza.append(
    registrar_cambio(
        tabla_afectada="clientes",
        descripcion="Duplicados exactos",
        filas_antes=filas_antes_del_cambio,
        filas_despues=len(clientes),
    )
)

filas_antes_del_cambio = len(productos)
productos = productos.drop_duplicates()
bitacora_limpieza.append(
    registrar_cambio(
        tabla_afectada="productos",
        descripcion="Duplicados exactos",
        filas_antes=filas_antes_del_cambio,
        filas_despues=len(productos),
    )
)

display(pd.DataFrame(bitacora_limpieza))

,tabla,cambio,filas_antes,filas_despues,filas_eliminadas,pct_eliminado
0,clientes,Duplicados exactos,3235,3211,24,0.74
1,productos,Duplicados exactos,6191,6083,108,1.74


**Qué columnas cambian entre las dos versiones de un mismo identificador.** Antes de elegir una versión hay que saber en qué se diferencian.

In [17]:
display(contar_columnas_que_difieren(clientes, "id_cliente"))
display(contar_columnas_que_difieren(productos, "id_producto"))

,ids_donde_cambia
ingreso_mensual,11


,ids_donde_cambia
tipo_producto,32


**Qué encontramos:** en los 11 clientes repetidos la **única** columna que cambia es `ingreso_mensual`; en los 32 productos repetidos la única que cambia es `tipo_producto`.

**Evidencia:** para cada identificador repetido se contó en cuántas columnas hay más de un valor distinto. Todas las demás columnas (edad, score, montos, fechas, atrasos) son iguales entre las dos versiones.

**Qué sugiere:** en productos no hay un dato en conflicto: es el mismo producto con el tipo escrito de dos maneras. En clientes sí hay conflicto, y es en la columna más delicada, porque el ingreso es el denominador del ratio cuota / ingreso.

In [18]:
# Miro las dos versiones de cada cliente repetido, con el ingreso pasado a número
clientes_repetidos = clientes[clientes["id_cliente"].duplicated(keep=False)].copy()
clientes_repetidos["ingreso_como_numero"] = clientes_repetidos["ingreso_mensual"].map(
    convertir_ingreso_a_numero
)
display(
    clientes_repetidos.sort_values("id_cliente")[
        ["id_cliente", "ingreso_mensual", "ingreso_como_numero"]
    ]
)

,id_cliente,ingreso_mensual,ingreso_como_numero
1461,1016215,26200,26200.0
1318,1016215,176572.00,176572.0
403,1045538,123282.00,123282.0
2075,1045538,41800.00,41800.0
2941,3712313,SIN DATO,NaN
2041,3712313,126334.00,126334.0
2446,3758810,217006.00,217006.0
1178,3758810,"90.300,00",90300.0
2886,5975365,"62.800,00",62800.0
2971,5975365,149249.00,149249.0


In [19]:
# Para elegir la versión buena miro qué tan comunes son los ingresos redondeados
# a la centena en el resto de la base (sin los 11 clientes en conflicto)
ingreso_resto_de_la_base = (
    clientes.loc[~clientes["id_cliente"].duplicated(keep=False), "ingreso_mensual"]
    .map(convertir_ingreso_a_numero)
    .dropna()
)
ingreso_resto_de_la_base = ingreso_resto_de_la_base[ingreso_resto_de_la_base > 0]
pct_redondeado = (ingreso_resto_de_la_base % REDONDEO_INGRESO == 0).mean()
print(
    f"Ingresos redondeados a la centena: {pct_redondeado:.1%} "
    f"de {len(ingreso_resto_de_la_base):,} clientes con ingreso numérico"
)

Ingresos redondeados a la centena: 100.0% de 3,035 clientes con ingreso numérico


**Criterio para los 11 clientes repetidos.** En el resto de la base, el 100% de los 3.035 clientes con ingreso numérico lo tienen redondeado a la centena (34.500, 55.400, 90.500...). En los 11 pares, una versión sigue ese patrón y la otra no (176.572, 123.282, 236.656...). Además, en los 9 pares donde las dos versiones tienen un número, la no redondeada es siempre más alta: entre 1,5 y 6,7 veces la otra.

Me quedo con la versión que es consistente con el resto de la base y elimino la versión con ingreso no redondeado. En dos casos la versión que queda tiene el ingreso vacío ("SIN DATO") o en 0: prefiero un ingreso desconocido, que se trata en los puntos 1.5 y 1.7, antes que un ingreso con un formato que no aparece en ningún otro cliente.

El archivo no trae una fecha de actualización por cliente, así que no puedo usar "la versión más nueva" como criterio. Quedarse con "la primera que aparece" tampoco es un criterio: es el orden del archivo.

In [20]:
# Elimino la versión del cliente repetido cuyo ingreso no está redondeado a la centena
ingreso_como_numero = clientes["ingreso_mensual"].map(convertir_ingreso_a_numero)
es_id_repetido = clientes["id_cliente"].duplicated(keep=False)
es_ingreso_no_redondeado = ingreso_como_numero.notna() & (
    ingreso_como_numero % REDONDEO_INGRESO != 0
)

filas_antes_del_cambio = len(clientes)
clientes = clientes[~(es_id_repetido & es_ingreso_no_redondeado)]
bitacora_limpieza.append(
    registrar_cambio(
        tabla_afectada="clientes",
        descripcion="Id repetido: versión con ingreso no redondeado",
        filas_antes=filas_antes_del_cambio,
        filas_despues=len(clientes),
    )
)

# Productos: las dos versiones son iguales salvo cómo está escrito el tipo de
# producto, así que da lo mismo cuál queda (el tipo se estandariza en el punto 1.5)
filas_antes_del_cambio = len(productos)
productos = productos.drop_duplicates(subset="id_producto")
bitacora_limpieza.append(
    registrar_cambio(
        tabla_afectada="productos",
        descripcion="Id repetido: mismo producto con el tipo escrito distinto",
        filas_antes=filas_antes_del_cambio,
        filas_despues=len(productos),
    )
)

display(pd.DataFrame(bitacora_limpieza))

,tabla,cambio,filas_antes,filas_despues,filas_eliminadas,pct_eliminado
0,clientes,Duplicados exactos,3235,3211,24,0.74
1,productos,Duplicados exactos,6191,6083,108,1.74
2,clientes,Id repetido: versión con ingreso no redondeado,3211,3200,11,0.34
3,productos,Id repetido: mismo producto con el tipo escrit...,6083,6051,32,0.53


In [21]:
# Verifico que no quede ningún identificador repetido
display(
    pd.DataFrame(
        {
            "clientes": contar_duplicados(clientes, "id_cliente"),
            "productos": contar_duplicados(productos, "id_producto"),
        }
    ).T
)

,filas,duplicados_exactos,ids_repetidos_en_total,ids_repetidos_con_datos_distintos
clientes,3200,0,0,0
productos,6051,0,0,0


**Qué encontramos:** después de las dos limpiezas clientes queda con 3.200 filas, una por cliente (se eliminó el 1,08% de la base: 24 + 11 filas), y productos con 6.051 filas, una por producto (se eliminó el 2,26%: 108 + 32 filas). No queda ningún identificador repetido.

**Evidencia:** la bitácora registra cada paso con las filas antes y después, y la tabla de verificación da 0 en todas las columnas de duplicados.

**Qué sugiere y qué falta:** el impacto sobre la base es chico, pero no era inocuo: con los clientes duplicados, cada producto de esos clientes se habría duplicado en el merge, inflando la cantidad de créditos y la mora. El criterio del ingreso redondeado es una inferencia sobre el formato, no una certeza: lo ideal sería pedirle al banco la fecha de última actualización de cada cliente para quedarse con la versión más reciente.

### 1.4 Faltantes por columna

Primero se cuenta lo que `isna()` detecta, y para cada columna con faltantes se muestran algunas filas afectadas. Los valores que el sistema usa para decir "no sé" (un 0, un -999, un "SIN DATO") todavía no aparecen acá: se buscan en el punto 1.5.

In [22]:
def resumir_faltantes(tabla):
    """Devuelve las columnas con faltantes, con la cantidad y el porcentaje de filas."""
    faltantes = tabla.isna().sum()
    resumen = pd.DataFrame(
        {"faltantes": faltantes, "pct_filas": (100 * faltantes / len(tabla)).round(2)}
    )
    return resumen[resumen["faltantes"] > 0].sort_values("faltantes", ascending=False)


def filtrar_filas_con_faltante(tabla, columna, columnas_a_mostrar, cantidad):
    """Devuelve algunas filas donde la columna está vacía, para mirarlas."""
    filas_con_faltante = tabla[tabla[columna].isna()]
    return filas_con_faltante[columnas_a_mostrar + [columna]].head(cantidad)

In [23]:
display(resumir_faltantes(clientes))
display(resumir_faltantes(productos))

,faltantes,pct_filas
antiguedad_laboral_meses,215,6.72
situacion_vivienda,148,4.62
score_clearing,140,4.38
ingreso_mensual,40,1.25
edad,10,0.31


,faltantes,pct_filas
fecha_primer_atraso_grave,3679,60.80
limite_credito,818,13.52
fecha_ultimo_reporte,557,9.21
saldo_maximo_historico,291,4.81
score_originacion,153,2.53


In [24]:
# Muestro algunas filas afectadas de cada columna con faltantes
for columna_con_faltantes in resumir_faltantes(clientes).index:
    display(
        filtrar_filas_con_faltante(
            tabla=clientes,
            columna=columna_con_faltantes,
            columnas_a_mostrar=["id_cliente", "sector_empleo"],
            cantidad=FILAS_A_MOSTRAR,
        )
    )

for columna_con_faltantes in resumir_faltantes(productos).index:
    display(
        filtrar_filas_con_faltante(
            tabla=productos,
            columna=columna_con_faltantes,
            columnas_a_mostrar=["id_producto", "tipo_producto"],
            cantidad=FILAS_A_MOSTRAR,
        )
    )

,id_cliente,sector_empleo,antiguedad_laboral_meses
12,4167950,Privado,NaN
21,5358153,Jubilado,NaN
24,5339081,Jubilado/a,NaN
70,9318381,Privado,NaN
106,8526263,Jubilado,NaN


,id_cliente,sector_empleo,situacion_vivienda
33,4222982,Informal,NaN
49,8274822,Privado,NaN
68,9977818,Jubilado/a,NaN
76,1642528,Privado,NaN
84,4410308,Jubilado,NaN


,id_cliente,sector_empleo,score_clearing
7,6917190,Jubilado,NaN
13,1943806,Jubilado,NaN
24,5339081,Jubilado/a,NaN
29,2452766,PRIVADO,NaN
36,8418433,Privado,NaN


,id_cliente,sector_empleo,ingreso_mensual
13,1943806,Jubilado,NaN
47,3093641,Privado,NaN
71,2405027,Independiente,NaN
94,2884087,Independiente,NaN
392,9579346,Independiente,NaN


,id_cliente,sector_empleo,edad
424,9328266,Jubilado,NaN
615,6192915,Changas,NaN
1204,8608315,Privado,NaN
1590,4705786,Jubilado,NaN
1599,9301745,Independiente,NaN


,id_producto,tipo_producto,fecha_primer_atraso_grave
0,900023,Adelanto de sueldo,NaT
3,500952,Préstamo automotor,NaT
4,505792,Adelanto de sueldo,NaT
5,500995,Adelanto sueldo,NaT
6,504245,Tarjeta de crédito,NaT


,id_producto,tipo_producto,limite_credito
1,900035,Tarjeta de crédito,NaN
3,500952,Préstamo automotor,NaN
4,505792,Adelanto de sueldo,NaN
5,500995,Adelanto sueldo,NaN
9,501337,Préstamo al consumo,NaN


,id_producto,tipo_producto,fecha_ultimo_reporte
1,900035,Tarjeta de crédito,NaT
9,501337,Préstamo al consumo,NaT
10,501276,Línea de crédito,NaT
19,504125,Adelanto de sueldo,NaT
29,503146,Crédito hipotecario,NaT


,id_producto,tipo_producto,saldo_maximo_historico
0,900023,Adelanto de sueldo,NaN
18,502242,Prestamo consumo,NaN
39,505705,Préstamo al consumo,NaN
72,503232,Préstamo automotor,NaN
118,501908,Tarjeta de crédito,NaN


,id_producto,tipo_producto,score_originacion
7,505185,Línea de crédito,NaN
37,500532,Línea de crédito,NaN
42,504792,Préstamo al consumo,NaN
45,900095,Préstamo automotor,NaN
215,502776,Prestamo consumo,NaN


In [25]:
# El enunciado dice que fecha_primer_atraso_grave queda "vacía si nunca pasó":
# lo verifico cruzándola con la cantidad de veces que el producto tuvo atraso grave
display(
    pd.crosstab(
        productos["fecha_primer_atraso_grave"].isna(),
        productos["veces_atraso_grave"] > 0,
        rownames=["fecha_vacia"],
        colnames=["tuvo_atraso_grave"],
    )
)

tuvo_atraso_grave,False,True
fecha_vacia,,
False,0,2372
True,3679,0


**Qué encontramos:** en clientes faltan datos en 5 columnas: `antiguedad_laboral_meses` (215 filas, 6,7%), `situacion_vivienda` (148, 4,6%), `score_clearing` (140, 4,4%), `ingreso_mensual` (40, 1,3%) y `edad` (10, 0,3%). En productos faltan en 5: `fecha_primer_atraso_grave` (3.679, 60,8%), `limite_credito` (818, 13,5%), `fecha_ultimo_reporte` (557, 9,2%), `saldo_maximo_historico` (291, 4,8%) y `score_originacion` (153, 2,5%).

**Evidencia:** `isna()` sobre las 3.200 filas de clientes y las 6.051 de productos, ya sin duplicados. El cruce confirma que el faltante más grande no es un faltante: las 3.679 fechas vacías son exactamente los 3.679 productos que nunca tuvieron atraso grave, y los 2.372 que sí lo tuvieron tienen todos su fecha.

**Qué sugiere:** `fecha_primer_atraso_grave` se deja como está, porque el vacío significa "nunca pasó". `limite_credito` y `fecha_ultimo_reporte` no se usan para medir la mora, así que no hace falta imputarlos. Las columnas que sí importan (ingreso, score y antigüedad) se miran por grupo en el punto 1.7, después de sumarles los "no sé" escondidos del punto 1.5.

### 1.5 Valores raros, imposibles y "no sé"

Tres cosas que `isna()` no detecta:

1. **Valores fuera del rango que admite el negocio** o códigos de "no sé": edad 0 o 202, score 0, plazo 999, días de atraso -999. Los rangos están en la constante `RANGOS_VALIDOS`, cada uno con su motivo.
2. **Inconsistencias entre columnas:** por ejemplo, alguien con más años en el trabajo que los que pudo haber trabajado.
3. **Categorías escritas de varias maneras:** "Montevideo", "montevideo", " Montevideo " y "MONTEVIDEO" son la misma.

Los valores imposibles se pasan a faltante (`NaN`) en lugar de borrar la fila: el resto de la fila sigue sirviendo, y un valor imposible no se puede "corregir" sin inventarlo.

In [26]:
def contar_fuera_de_rango(tabla, rangos):
    """Cuenta los valores con dato que caen fuera del rango válido de cada columna."""
    filas_resumen = {}
    for columna, (minimo, maximo) in rangos.items():
        if columna not in tabla.columns:
            continue
        con_dato = tabla[columna].dropna()
        fuera = con_dato[~con_dato.between(minimo, maximo)]
        filas_resumen[columna] = {
            "con_dato": len(con_dato),
            "fuera_de_rango": len(fuera),
            "pct_filas": round(100 * len(fuera) / len(tabla), 2),
            "valores_mas_frecuentes": fuera.value_counts().head(3).index.tolist(),
        }
    return pd.DataFrame(filas_resumen).T


def pasar_fuera_de_rango_a_faltante(tabla, rangos):
    """Devuelve una copia de la tabla con los valores fuera de rango como NaN."""
    tabla_limpia = tabla.copy()
    for columna, (minimo, maximo) in rangos.items():
        if columna not in tabla_limpia.columns:
            continue
        fuera = tabla_limpia[columna].notna() & ~tabla_limpia[columna].between(
            minimo, maximo
        )
        tabla_limpia[columna] = tabla_limpia[columna].mask(fuera)
    return tabla_limpia


def comparar_faltantes(tabla_antes, tabla_despues, columnas):
    """Compara los faltantes de cada columna antes y después de una limpieza."""
    faltantes_antes = tabla_antes[columnas].isna().sum()
    faltantes_despues = tabla_despues[columnas].isna().sum()
    return pd.DataFrame(
        {
            "faltantes_antes": faltantes_antes,
            "faltantes_despues": faltantes_despues,
            "pct_base_afectada": (
                100 * (faltantes_despues - faltantes_antes) / len(tabla_antes)
            ).round(2),
        }
    )

In [27]:
display(contar_fuera_de_rango(clientes, RANGOS_VALIDOS))
display(contar_fuera_de_rango(productos, RANGOS_VALIDOS))

,con_dato,fuera_de_rango,pct_filas,valores_mas_frecuentes
edad,3190,52,1.62,"[0.0, 17.0, 16.0]"
antiguedad_laboral_meses,2985,23,0.72,"[-16.0, -20.0, -25.0]"
score_clearing,3060,55,1.72,[0.0]


,con_dato,fuera_de_rango,pct_filas,valores_mas_frecuentes
score_originacion,5898,62,1.02,[0.0]
plazo_meses,6051,28,0.46,"[999, 0]"
monto_otorgado,6051,12,0.2,"[-7719.68, -75732.74, -189811.75]"
dias_atraso_acumulados,6051,183,3.02,[-999]


**Qué encontramos:** en clientes, 52 edades fuera de rango (20 en 0, 22 entre 16 y 17 años y 10 entre 191 y 202), 23 antigüedades laborales negativas y 55 scores del Clearing en 0. En productos, 28 plazos imposibles (14 en 0 y 14 en 999), 62 scores de originación en 0, 12 montos otorgados negativos y 183 productos con -999 días de atraso.

**Evidencia:** cada columna se comparó contra su rango válido en `RANGOS_VALIDOS`. El 999 y el -999 son siempre el mismo número, y el 0 en el score es imposible en una escala que arranca en 300: son códigos que el sistema usa para decir "no sé", no mediciones.

**Qué sugiere:** se pasan a faltante. Si quedaran, el -999 bajaría el promedio de días de atraso y el 999 subiría el plazo promedio, sin que nadie lo note.

In [28]:
# Paso a faltante los valores fuera de rango y reporto cuánto de la base cambió
clientes_antes = clientes
clientes = pasar_fuera_de_rango_a_faltante(clientes, RANGOS_VALIDOS)
display(
    comparar_faltantes(
        tabla_antes=clientes_antes,
        tabla_despues=clientes,
        columnas=["edad", "antiguedad_laboral_meses", "score_clearing"],
    )
)

productos_antes = productos
productos = pasar_fuera_de_rango_a_faltante(productos, RANGOS_VALIDOS)
display(
    comparar_faltantes(
        tabla_antes=productos_antes,
        tabla_despues=productos,
        columnas=[
            "plazo_meses",
            "score_originacion",
            "monto_otorgado",
            "dias_atraso_acumulados",
        ],
    )
)

,faltantes_antes,faltantes_despues,pct_base_afectada
edad,10,62,1.62
antiguedad_laboral_meses,215,238,0.72
score_clearing,140,195,1.72


,faltantes_antes,faltantes_despues,pct_base_afectada
plazo_meses,0,28,0.46
score_originacion,153,215,1.02
monto_otorgado,0,12,0.20
dias_atraso_acumulados,0,183,3.02


**Inconsistencias entre columnas.** Un valor puede estar dentro de su rango y aun así contradecir a otra columna. Se revisan tres cruces.

In [29]:
# Antigüedad laboral mayor que los meses que la persona pudo haber trabajado
meses_trabajables = (clientes["edad"] - EDAD_MINIMA_PARA_TRABAJAR) * MESES_POR_ANIO
antiguedad_imposible = clientes["antiguedad_laboral_meses"] > meses_trabajables

# Cuotas pagas en término por encima del total de cuotas pactadas
cuotas_imposibles = productos["cuotas_pagas_en_termino"] > productos["plazo_meses"]

# Producto con atraso grave (más de 60 días) pero 60 días de atraso o menos en total
atraso_grave_sin_dias = (productos["veces_atraso_grave"] > 0) & (
    productos["dias_atraso_acumulados"] <= 60  # 60 días: el límite del atraso grave
)

display(
    pd.DataFrame(
        {
            "filas": [
                antiguedad_imposible.sum(),
                cuotas_imposibles.sum(),
                atraso_grave_sin_dias.sum(),
            ],
            "pct_tabla": [
                round(100 * antiguedad_imposible.mean(), 2),
                round(100 * cuotas_imposibles.mean(), 2),
                round(100 * atraso_grave_sin_dias.mean(), 2),
            ],
        },
        index=[
            "clientes: antigüedad mayor que (edad - 14) años",
            "productos: cuotas pagas mayores que el plazo",
            "productos: atraso grave con 60 días acumulados o menos",
        ],
    )
)

,filas,pct_tabla
clientes: antigüedad mayor que (edad - 14) años,16,0.50
productos: cuotas pagas mayores que el plazo,0,0.00
productos: atraso grave con 60 días acumulados o menos,193,3.19


In [30]:
# La antigüedad imposible la paso a faltante: no sé cuál de las dos columnas está mal,
# pero la edad la uso más que la antigüedad, así que la que descarto es la antigüedad
clientes_antes = clientes
clientes = clientes.assign(
    antiguedad_laboral_meses=clientes["antiguedad_laboral_meses"].mask(
        antiguedad_imposible
    )
)
display(
    comparar_faltantes(
        tabla_antes=clientes_antes,
        tabla_despues=clientes,
        columnas=["antiguedad_laboral_meses"],
    )
)

,faltantes_antes,faltantes_despues,pct_base_afectada
antiguedad_laboral_meses,238,254,0.5


**Qué encontramos:** hay 16 clientes con más antigüedad laboral que los años que pudieron trabajar desde los 14 (0,5% de la base) y 193 productos que figuran con atraso grave pero con 60 días de atraso acumulados o menos (3,2%). No queda ningún producto con más cuotas pagas que cuotas pactadas: los 14 que había eran los de plazo 0, que ya se pasaron a faltante.

**Evidencia:** los tres cruces se calcularon después de pasar a faltante los valores fuera de rango, así que no los generan ni el 999 ni el -999 ni las edades imposibles (antes de limpiar la edad, el cruce de antigüedad daba 44).

**Qué sugiere:** la antigüedad imposible se pasa a faltante. Los 193 productos se dejan como están pero se anotan: no afectan la definición de mora, que se apoya en `veces_atraso_grave` y en la fecha del primer atraso grave (que coinciden entre sí al 100%, punto 1.4). Los 193 productos son una pista de un problema operativo: el contador de días de atraso y el de atrasos graves no están sincronizados, y habría que preguntarle a Sistemas cómo se calcula cada uno.

**Categorías escritas de varias maneras.** Se estandarizan en dos pasos: primero se ignoran mayúsculas, tildes y espacios de más (así "MONTEVIDEO", " Montevideo " y "montevideo" quedan iguales), y después se aplican los sinónimos de `SINONIMOS_CATEGORIAS` para las escrituras que son otra palabra ("Alquiler" por "Alquilada", "Changas" por "Informal").

In [31]:
def normalizar_texto(texto):
    """Pasa a minúscula, saca tildes y espacios de más, para comparar escrituras."""
    sin_tildes = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore")
    return sin_tildes.decode("ascii").strip().lower()


def estandarizar_categoria(serie, categorias_validas, sinonimos):
    """Reemplaza cada escritura por su categoría válida (NaN si no la reconoce)."""
    equivalencias = {normalizar_texto(valor): valor for valor in categorias_validas}
    equivalencias.update(
        {normalizar_texto(escrito): valido for escrito, valido in sinonimos.items()}
    )
    return serie.map(
        lambda texto: equivalencias.get(normalizar_texto(texto)), na_action="ignore"
    )


def resumir_estandarizacion(serie_antes, serie_despues):
    """Cuenta las categorías y las filas que cambiaron al estandarizar una columna."""
    fila_modificada = serie_antes.notna() & (serie_despues != serie_antes)
    return {
        "categorias_antes": serie_antes.nunique(),
        "categorias_despues": serie_despues.nunique(),
        "filas_modificadas": fila_modificada.sum(),
        "pct_filas_modificadas": round(100 * fila_modificada.mean(), 2),
        "sin_reconocer": (serie_antes.notna() & serie_despues.isna()).sum(),
    }

In [32]:
# Estandarizo cada columna categórica y verifico que no quede ninguna sin reconocer
columnas_a_estandarizar = [
    (clientes, "departamento"),
    (clientes, "sector_empleo"),
    (clientes, "situacion_vivienda"),
    (clientes, "canal_originacion"),
    (productos, "tipo_producto"),
]
resumen_estandarizacion = {}
for tabla_a_limpiar, columna_categorica in columnas_a_estandarizar:
    estandarizada = estandarizar_categoria(
        serie=tabla_a_limpiar[columna_categorica],
        categorias_validas=CATEGORIAS_VALIDAS[columna_categorica],
        sinonimos=SINONIMOS_CATEGORIAS[columna_categorica],
    )
    resumen_estandarizacion[columna_categorica] = resumir_estandarizacion(
        tabla_a_limpiar[columna_categorica], estandarizada
    )
    tabla_a_limpiar[columna_categorica] = estandarizada

display(pd.DataFrame(resumen_estandarizacion).T)

,categorias_antes,categorias_despues,filas_modificadas,pct_filas_modificadas,sin_reconocer
departamento,55.0,19.0,654.0,20.44,0.0
sector_empleo,10.0,5.0,518.0,16.19,0.0
situacion_vivienda,7.0,4.0,330.0,10.31,0.0
canal_originacion,10.0,5.0,528.0,16.50,0.0
tipo_producto,10.0,6.0,734.0,12.13,0.0


In [33]:
display(contar_categorias(clientes, CATEGORIAS_ESPERADAS))
display(contar_categorias(productos, CATEGORIAS_ESPERADAS))

,categorias_en_archivo,categorias_esperadas
id_cliente,3200,<NA>
ingreso_mensual,2072,<NA>
departamento,19,19
nivel_educativo,6,<NA>
estado_civil,5,5
canal_originacion,5,5
sector_empleo,5,5
situacion_vivienda,4,4
segmento_comercial,3,3
sexo,2,2


,categorias_en_archivo,categorias_esperadas
id_producto,6051,<NA>
id_cliente,3296,<NA>
tipo_producto,6,6
moneda,3,3


**Qué encontramos:** las 5 columnas quedaron con la cantidad de categorías que dice el enunciado: departamento pasó de 55 a 19, sector y canal de 10 a 5, vivienda de 7 a 4 y tipo de producto de 10 a 6. Se modificó la escritura de entre el 10% y el 20% de las filas según la columna (en departamento, 654 clientes), y ninguna quedó sin reconocer.

**Evidencia:** la tabla de estandarización compara cada columna antes y después; el conteo de categorías se repite con la misma función del punto 1.2 y ahora coincide con lo esperado en todas las columnas. Los 148 vacíos de `situacion_vivienda` siguen siendo vacíos: estandarizar no inventa datos.

**Qué sugiere:** sin este paso, Montevideo aparecería repartido en 4 filas de cualquier tabla por departamento y su peso real quedaría subestimado. Falta todavía el "no sé" del ingreso ("S/D", "SIN DATO", 0 y -999), que se trata junto con sus formatos en el punto 1.6.

### 1.6 Columnas con dos unidades o dos monedas

Se revisan tres columnas:

- **`tasa_nominal_anual`**: el promedio (4,42) es 6 veces la mediana (0,71). La mayoría viene como fracción (0,71 = 71% anual) y una parte en porcentaje (71 = 71% anual).
- **`ingreso_mensual`**: viene en tres formatos de escritura ("34500.00", "$ 55.400", "29.100,00") y con códigos de "no sé". Hay que confirmar además que esté todo en pesos.
- **`monto_otorgado` y `cuota_mensual`**: están en la moneda del contrato (pesos, dólares o UI). Se pasan a pesos en el punto 2.2 con el tipo de cambio del mes de otorgamiento.

In [34]:
# Para elegir el umbral miro, por tipo de producto, la tasa más alta escrita como
# fracción y la más baja escrita como porcentaje: tiene que haber un salto claro
es_porcentaje = productos["tasa_nominal_anual"] > UMBRAL_TASA_EN_PORCENTAJE
display(
    pd.DataFrame(
        {
            "maxima_como_fraccion": productos[~es_porcentaje]
            .groupby("tipo_producto")["tasa_nominal_anual"]
            .max(),
            "minima_como_porcentaje": productos[es_porcentaje]
            .groupby("tipo_producto")["tasa_nominal_anual"]
            .min(),
        }
    )
)

,maxima_como_fraccion,minima_como_porcentaje
tipo_producto,,
Adelanto de sueldo,1.0998,75.37
Crédito hipotecario,0.0847,5.72
Línea de crédito,0.8496,50.12
Préstamo al consumo,0.7197,42.80
Préstamo automotor,0.1600,9.01
Tarjeta de crédito,0.9799,68.29


In [35]:
# Divido por 100 las tasas escritas en porcentaje para que todas queden como fracción
tasa_antes = productos["tasa_nominal_anual"]
productos["tasa_nominal_anual"] = tasa_antes.mask(
    es_porcentaje, tasa_antes / PORCENTAJE_A_FRACCION
)
display(
    pd.DataFrame(
        {
            "antes": tasa_antes.describe(),
            "despues": productos["tasa_nominal_anual"].describe(),
        }
    ).round(3)
)
print(
    f"Tasas convertidas: {es_porcentaje.sum()} de {len(productos):,} "
    f"({es_porcentaje.mean():.1%} de la base)"
)

,antes,despues
count,6051.000,6051.000
mean,4.440,0.643
std,16.338,0.271
min,0.055,0.055
25%,0.528,0.516
50%,0.714,0.700
75%,0.874,0.842
max,109.340,1.100


Tasas convertidas: 370 de 6,051 (6.1% de la base)


**Qué encontramos:** 370 tasas (6,1% de los productos) venían en porcentaje. Después de dividirlas por 100, el promedio pasó de 4,44 a 0,64 y la mediana de 0,71 a 0,70, y la tasa más alta quedó en 1,10 (110% anual): ahora promedio y mediana cuentan la misma historia.

**Evidencia:** en todos los tipos de producto hay un salto claro entre la tasa más alta escrita como fracción (como mucho 1,10, en adelantos de sueldo) y la más baja escrita como porcentaje (como poco 5,72, en hipotecarios). El umbral de 1,5 cae en el medio de ese salto en todos los tipos.

**Qué sugiere:** la tasa ya se puede comparar entre productos. El problema está repartido parejo en los cuatro años (entre 5,6% y 7,6% de los productos de cada año), así que parece un error de carga y no un cambio de sistema en una fecha.

In [36]:
def clasificar_formato_ingreso(ingreso_texto):
    """Dice en cuál de los formatos del archivo está escrito un ingreso."""
    if pd.isna(ingreso_texto):
        return "vacío"
    if "," in ingreso_texto:
        return "coma decimal (29.100,00)"
    if "$" in ingreso_texto or re.match(PATRON_MILES_CON_PUNTO, ingreso_texto):
        return "punto de miles ($ 55.400)"
    if re.match(r"^-?[\d.]+$", ingreso_texto):
        return "número simple (34500.00)"
    return "texto de no sé (S/D, SIN DATO)"

In [37]:
# Convierto el ingreso a número y reporto cuántos clientes hay en cada formato
formato_ingreso = clientes["ingreso_mensual"].map(clasificar_formato_ingreso)
ingreso_numerico = clientes["ingreso_mensual"].map(convertir_ingreso_a_numero)
display(
    pd.DataFrame(
        {
            "clientes": formato_ingreso.value_counts(),
            "pct_base": (100 * formato_ingreso.value_counts(normalize=True)).round(2),
            "sin_numero_despues_de_convertir": ingreso_numerico.isna()
            .groupby(formato_ingreso)
            .sum(),
        }
    )
)

# Un ingreso de 0 o negativo (-999) no es un ingreso: es otro código de "no sé"
ingreso_no_valido = ingreso_numerico <= 0
print(
    f"Ingresos en 0 o negativos: {ingreso_no_valido.sum()} "
    f"(valores: {sorted(ingreso_numerico[ingreso_no_valido].unique())})"
)
clientes["ingreso_mensual"] = ingreso_numerico.mask(ingreso_no_valido)
print(
    f"Clientes sin ingreso usable: {clientes['ingreso_mensual'].isna().sum()} "
    f"de {len(clientes):,} ({clientes['ingreso_mensual'].isna().mean():.1%})"
)
display(clientes["ingreso_mensual"].describe().round(0))

,clientes,pct_base,sin_numero_despues_de_convertir
ingreso_mensual,,,
"coma decimal (29.100,00)",886,27.69,0
número simple (34500.00),1701,53.16,0
punto de miles ($ 55.400),502,15.69,0
"texto de no sé (S/D, SIN DATO)",71,2.22,71
vacío,40,1.25,40


Ingresos en 0 o negativos: 45 (valores: [np.float64(-999.0), np.float64(0.0)])
Clientes sin ingreso usable: 156 de 3,200 (4.9%)


count      3044.0
mean      66188.0
std       35812.0
min       16000.0
25%       41500.0
50%       58100.0
75%       83500.0
max      371800.0
Name: ingreso_mensual, dtype: float64

**Qué encontramos:** de los 3.200 clientes, el ingreso de 1.701 venía como número simple, el de 886 con coma decimal y el de 502 con punto de miles; esos 3.089 se convirtieron sin perder ninguno. Pero 45 de ellos no son un ingreso (21 ceros y 24 "-999"), y además hay 71 "S/D" o "SIN DATO" y 40 vacíos: en total, 156 clientes (4,9%) sin un ingreso usable.

**Evidencia:** cada fila se clasificó por su formato y se contó cuántas quedaron sin número después de convertir: solo las que no tenían un número para empezar. El ingreso válido va de $16.000 a $371.800, con mediana de $58.100: no hay ningún valor tan chico como para estar escrito en dólares (el más bajo, 16.000 dólares, serían más de $600.000 por mes), así que está todo en pesos.

**Qué sugiere:** si se hubiera usado `pd.to_numeric(errors="coerce")` sin limpiar, los 1.388 ingresos con coma decimal o punto de miles se habrían perdido (43% de la base), y en cambio los 0 y los -999 habrían quedado como si fueran ingresos. Qué hacer con los 156 sin ingreso se decide en el punto 1.7.

### 1.7 ¿Los faltantes se concentran en algún grupo? Qué hacer con los clientes sin ingreso

Si el ingreso falta al azar, imputarlo con la mediana es defendible. Si falta más en un grupo, imputar con la mediana general le pone a ese grupo un ingreso que no es el suyo, y si además ese grupo es más riesgoso, el sesgo cae justo donde importa.

Para medir si "más riesgoso" se usa un indicador provisorio: si el cliente tuvo **algún producto con atraso grave** (más de 60 días). La definición de mora se discute en el punto 2.1.

In [38]:
def calcular_tasa_por_grupo(tabla, columna_grupo, columna_indicadora):
    """Devuelve, por grupo, la cantidad de filas y el % con el indicador verdadero."""
    return (
        tabla.groupby(columna_grupo)[columna_indicadora]
        .agg(filas="size", pct=lambda indicador: round(100 * indicador.mean(), 1))
        .sort_values("pct", ascending=False)
    )

In [39]:
# Indicador por cliente: ¿alguno de sus productos tuvo atraso grave?
cliente_con_atraso_grave = (
    productos.groupby("id_cliente")["veces_atraso_grave"]
    .max()
    .gt(0)
    .rename("tuvo_atraso_grave")
    .reset_index()
)

filas_antes_del_cambio = len(clientes)
clientes = clientes.merge(cliente_con_atraso_grave, on="id_cliente", how="left")
print(f"Filas de clientes antes del merge: {filas_antes_del_cambio:,}")
print(f"Filas de clientes después del merge: {len(clientes):,}")
print(f"Clientes sin productos: {clientes['tuvo_atraso_grave'].isna().sum()}")

clientes["sin_ingreso"] = clientes["ingreso_mensual"].isna()

Filas de clientes antes del merge: 3,200
Filas de clientes después del merge: 3,200
Clientes sin productos: 0


In [40]:
display(calcular_tasa_por_grupo(clientes, "sector_empleo", "sin_ingreso"))
display(calcular_tasa_por_grupo(clientes, "tiene_cuenta_sueldo", "sin_ingreso"))
display(calcular_tasa_por_grupo(clientes, "canal_originacion", "sin_ingreso"))

,filas,pct
sector_empleo,,
Informal,213,17.8
Independiente,498,12.7
Jubilado,891,2.9
Privado,1130,2.0
Público,468,1.3


,filas,pct
tiene_cuenta_sueldo,,
No,1747,6.4
Si,1453,3.0


,filas,pct
canal_originacion,,
Telemarketing,261,6.1
App,456,5.9
Sucursal,1266,4.9
Comercio adherido,703,4.8
Web,514,3.3


In [41]:
display(calcular_tasa_por_grupo(clientes, "sin_ingreso", "tuvo_atraso_grave"))
display(clientes.groupby("sin_ingreso")["score_clearing"].median().to_frame())

,filas,pct
sin_ingreso,,
True,156,66.7
False,3044,53.7


,score_clearing
sin_ingreso,
False,664.0
True,629.0


**Qué encontramos:** el ingreso no falta al azar. Falta en el 17,8% de los informales y el 12,7% de los independientes, contra el 1,3% de los públicos, el 2,0% de los privados y el 2,9% de los jubilados. También falta el doble en los clientes sin cuenta sueldo (6,4%) que en los que la tienen (3,0%). Por canal, en cambio, va de 3,3% a 6,1%: un rango angosto.

**Evidencia:** tasa de ingreso faltante por grupo sobre los 3.200 clientes. Y lo más importante: el 66,7% de los 156 clientes sin ingreso tuvo algún atraso grave, contra el 53,7% de los que sí lo informaron (13 puntos más), y su score mediano es 35 puntos más bajo (629 contra 664).

**Decisión: no imputar el ingreso.** Los clientes sin ingreso son justamente los trabajadores sin recibo de sueldo, y son más riesgosos. Ponerles la mediana general ($58.100) les asignaría el ingreso de un empleado formal y un ratio cuota / ingreso más sano que el real: el sesgo caería en el grupo más riesgoso. No se borran (son el 4,9% de la base y se usan en todo el análisis que no necesita el ingreso) y en los tramos de ingreso quedan como una categoría propia, **"No informado"**, porque no informar el ingreso ya es una señal de riesgo. Del análisis del ratio cuota / ingreso quedan afuera, porque no se puede calcular.

**Qué faltaría saber:** si el banco tiene algún dato de ingreso alternativo para estos clientes (por ejemplo, los depósitos de su cuenta), que permitiría estimarlo sin inventarlo.

In [42]:
# Bitácora completa de la etapa 1: lo que se eliminó y lo que se pasó a faltante
display(pd.DataFrame(bitacora_limpieza))
display(resumir_faltantes(clientes))
display(resumir_faltantes(productos))

,tabla,cambio,filas_antes,filas_despues,filas_eliminadas,pct_eliminado
0,clientes,Duplicados exactos,3235,3211,24,0.74
1,productos,Duplicados exactos,6191,6083,108,1.74
2,clientes,Id repetido: versión con ingreso no redondeado,3211,3200,11,0.34
3,productos,Id repetido: mismo producto con el tipo escrit...,6083,6051,32,0.53


,faltantes,pct_filas
antiguedad_laboral_meses,254,7.94
score_clearing,195,6.09
ingreso_mensual,156,4.88
situacion_vivienda,148,4.62
edad,62,1.94


,faltantes,pct_filas
fecha_primer_atraso_grave,3679,60.80
limite_credito,818,13.52
fecha_ultimo_reporte,557,9.21
saldo_maximo_historico,291,4.81
score_originacion,215,3.55
dias_atraso_acumulados,183,3.02
plazo_meses,28,0.46
monto_otorgado,12,0.20


**Cierre de la etapa 1.** Se eliminaron 35 filas de clientes (1,08%) y 140 de productos (2,26%), todas duplicadas. Ninguna otra fila se borró: los valores imposibles y los "no sé" se pasaron a faltante y quedaron a la vista en la tabla de faltantes final, que es la base sobre la que se construyen las variables de la etapa 2.

## 2. Creación de variables

### 2.1 Qué es un cliente en mora

"Cliente que se atrasa" no es una columna del archivo. Hay varias definiciones defendibles según **cuánto** atraso se cuenta y **en qué período**:

| Definición | Qué cuenta |
|---|---|
| Atraso de cualquier tamaño | Alguna vez tuvo un atraso, aunque sea de 1 día |
| Atraso de más de 30 días | Alguna vez llegó a atraso medio o grave |
| **Atraso grave (más de 60 días)** | Alguna vez llegó a atraso grave |
| Atraso grave en los primeros 6 meses | Llegó a atraso grave dentro de los 6 meses posteriores al otorgamiento, contando solo créditos que ya cumplieron 6 meses |

Se calcula cada una a nivel de producto y a nivel de cliente. Un **cliente está en mora** si al menos uno de sus productos lo está.

In [43]:
def calcular_tasas_de_mora(tabla, columna_mora):
    """Devuelve el % de productos y de clientes que cumplen la definición de mora."""
    mora_por_cliente = tabla.groupby("id_cliente")[columna_mora].max()
    return {
        "productos": len(tabla),
        "pct_productos": round(100 * tabla[columna_mora].mean(), 1),
        "clientes": len(mora_por_cliente),
        "pct_clientes": round(100 * mora_por_cliente.mean(), 1),
    }

In [44]:
# Construyo las cuatro definiciones de mora candidatas, a nivel de producto
productos["atraso_cualquiera"] = (
    productos[["veces_atraso_leve", "veces_atraso_medio", "veces_atraso_grave"]]
    .gt(0)
    .any(axis=1)
)
productos["atraso_mas_de_30_dias"] = (
    productos[["veces_atraso_medio", "veces_atraso_grave"]].gt(0).any(axis=1)
)
productos["en_mora"] = productos["veces_atraso_grave"] > 0

# Para la ventana de 6 meses solo sirven los créditos que ya cumplieron 6 meses al corte
fin_de_ventana = productos["fecha_alta_producto"] + VENTANA_MORA
productos["cumplio_ventana"] = fin_de_ventana <= FECHA_CORTE
productos["mora_6_meses"] = productos["fecha_primer_atraso_grave"] <= fin_de_ventana

productos_con_ventana = productos[productos["cumplio_ventana"]]
display(
    pd.DataFrame(
        {
            "Atraso de cualquier tamaño": calcular_tasas_de_mora(
                productos, "atraso_cualquiera"
            ),
            "Atraso de más de 30 días": calcular_tasas_de_mora(
                productos, "atraso_mas_de_30_dias"
            ),
            "Atraso grave (más de 60 días)": calcular_tasas_de_mora(
                productos, "en_mora"
            ),
            "Atraso grave en los primeros 6 meses": calcular_tasas_de_mora(
                productos_con_ventana, "mora_6_meses"
            ),
        }
    ).T
)

# Qué parte de los atrasos graves ocurre dentro de los primeros 6 meses
atrasos_graves_con_ventana = productos_con_ventana[productos_con_ventana["en_mora"]]
print(
    f"Atrasos graves que ocurren en los primeros {MESES_VENTANA_MORA} meses: "
    f"{atrasos_graves_con_ventana['mora_6_meses'].mean():.1%} "
    f"de {len(atrasos_graves_con_ventana):,}"
)

,productos,pct_productos,clientes,pct_clientes
Atraso de cualquier tamaño,6051.0,70.6,3296.0,81.7
Atraso de más de 30 días,6051.0,55.8,3296.0,69.7
Atraso grave (más de 60 días),6051.0,39.2,3296.0,53.8
Atraso grave en los primeros 6 meses,5589.0,18.7,3166.0,28.6


Atrasos graves que ocurren en los primeros 6 meses: 45.1% de 2,312


**Qué encontramos:** la tasa global de mora cambia muchísimo según la definición. A nivel de cliente va del 81,7% (algún atraso, aunque sea de 1 día) al 69,7% (más de 30 días), al 53,8% (atraso grave) y al 28,6% (atraso grave en los primeros 6 meses). A nivel de producto: 70,6%, 55,8%, 39,2% y 18,7%.

**Evidencia:** 6.051 productos de 3.296 clientes (todavía incluye los 96 clientes que no están en clientes.csv, que se tratan en el punto 2.3). La ventana de 6 meses se calcula sobre los 5.589 productos que ya cumplieron 6 meses al 30/06/2026, y captura el 45,1% de los atrasos graves de esos productos.

**Decisión.** Un **cliente está en mora** si al menos uno de sus productos llegó a **atraso grave (más de 60 días)**:

- Un atraso de cualquier tamaño le pasó a 8 de cada 10 clientes: no separa a nadie, y 1 día de atraso puede ser un olvido.
- Más de 30 días incluye atrasos que después se regularizan.
- Más de 60 días es lo que el propio banco llama atraso grave, y es la única definición que trae fecha (`fecha_primer_atraso_grave`), así que permite saber **cuándo** pasó.

Para **comparar grupos y trimestres** se usa la versión de los primeros 6 meses (`mora_6_meses`). Un crédito de 2023 tuvo tres años para atrasarse y uno de fines de 2025 solo seis meses: comparar "alguna vez" mide antigüedad, no calidad (es la censura del Tema 9). Se eligen 6 meses porque es la ventana más larga que deja adentro todo 2025, que es el período que preguntó el Directorio; con 12 meses habría que dejar afuera todo lo otorgado después de junio de 2025. El costo es que la tasa es más baja que la final (captura el 45% de los atrasos graves), pero es la misma vara para todos.

### 2.2 Montos y cuotas en pesos, con el tipo de cambio del mes de otorgamiento

`monto_otorgado` y `cuota_mensual` están en la moneda del contrato: sumar o comparar un préstamo en dólares con uno en pesos no tiene sentido. Se pasan a pesos con la cotización **del mes en que se otorgó el crédito**, que es la que reflejaba el esfuerzo de pago del cliente en ese momento. El peso queda igual; el dólar se multiplica por el tipo de cambio y la UI por su valor en pesos.

In [45]:
def calcular_factor_a_pesos(tabla, cotizacion_por_moneda):
    """Devuelve, por fila, cuántos pesos vale una unidad de la moneda del contrato."""
    factor = pd.Series(1.0, index=tabla.index)  # los pesos quedan como están
    for moneda, columna_cotizacion in cotizacion_por_moneda.items():
        es_moneda = tabla["moneda"] == moneda
        factor = factor.mask(es_moneda, tabla[columna_cotizacion])
    return factor

In [46]:
# Pego a cada producto la cotización del mes en que se otorgó
productos["mes_otorgamiento"] = (
    productos["fecha_alta_producto"].dt.to_period("M").dt.to_timestamp()
)
filas_antes_del_cambio = len(productos)
productos = productos.merge(
    tipo_cambio, left_on="mes_otorgamiento", right_on="mes", how="left"
)
print(f"Productos antes del merge: {filas_antes_del_cambio:,}")
print(f"Productos después del merge: {len(productos):,}")
print(f"Productos sin cotización: {productos['tipo_cambio_usd'].isna().sum()}")

factor_a_pesos = calcular_factor_a_pesos(productos, COTIZACION_POR_MONEDA)
productos["monto_pesos"] = productos["monto_otorgado"] * factor_a_pesos
productos["cuota_pesos"] = productos["cuota_mensual"] * factor_a_pesos

display(
    productos.groupby("moneda")[
        ["monto_otorgado", "monto_pesos", "cuota_mensual", "cuota_pesos"]
    ]
    .median()
    .round(0)
)

Productos antes del merge: 6,051
Productos después del merge: 6,051
Productos sin cotización: 0


,monto_otorgado,monto_pesos,cuota_mensual,cuota_pesos
moneda,,,,
UI,572557.0,3505906.0,4950.0,30147.0
USD,19202.0,794275.0,583.0,24442.0
UYU,129423.0,129423.0,12744.0,12744.0


**Qué encontramos:** todos los productos encontraron su cotización. En pesos, el préstamo mediano en dólares pasa de 19.202 a $794.275 y el mediano en UI de 572.557 a $3.505.906 (son sobre todo hipotecarios), contra $129.423 del mediano en pesos.

**Evidencia:** merge de los 6.051 productos con las 42 cotizaciones mensuales por el mes de otorgamiento: 6.051 filas antes y después, y 0 productos sin tipo de cambio.

**Qué sugiere:** sin convertir, un préstamo de 19.000 dólares parecería siete veces más chico que uno de 129.000 pesos, cuando en realidad es seis veces más grande. A partir de acá todos los montos y cuotas se usan en pesos (`monto_pesos`, `cuota_pesos`).

### 2.3 Ratio cuota / ingreso

Es la parte del ingreso mensual del cliente que se lleva la cuota del crédito. Se calcula para cada crédito, con la cuota en pesos del punto 2.2 y el ingreso limpio del punto 1.6, porque es lo que el banco puede controlar al momento de otorgar: "¿esta cuota, para este ingreso, es pagable?".

Para eso hay que unir productos con clientes. Hay productos de clientes que no están en la tabla de clientes (punto 1.2): se cuentan y se deciden antes de seguir.

In [47]:
# Uno cada crédito con los datos de su cliente y verifico el merge
filas_antes_del_cambio = len(productos)
creditos = productos.merge(clientes, on="id_cliente", how="left")
print(f"Créditos antes del merge: {filas_antes_del_cambio:,}")
print(f"Créditos después del merge: {len(creditos):,}")
creditos_sin_cliente = creditos["sexo"].isna()
print(
    f"Créditos de clientes que no están en clientes.csv: {creditos_sin_cliente.sum()} "
    f"({creditos_sin_cliente.mean():.1%}), de "
    f"{creditos.loc[creditos_sin_cliente, 'id_cliente'].nunique()} clientes distintos"
)
display(creditos.loc[creditos_sin_cliente, ["en_mora", "mora_6_meses"]].mean().round(3))

Créditos antes del merge: 6,051
Créditos después del merge: 6,051
Créditos de clientes que no están en clientes.csv: 96 (1.6%), de 96 clientes distintos


en_mora         0.365
mora_6_meses    0.198
dtype: float64

In [48]:
# Saco los créditos sin cliente: sin sus datos no puedo calcular el ratio ni el perfil
filas_antes_del_cambio = len(creditos)
creditos = creditos[~creditos_sin_cliente].copy()
bitacora_limpieza.append(
    registrar_cambio(
        tabla_afectada="créditos",
        descripcion="Créditos de clientes que no están en clientes.csv",
        filas_antes=filas_antes_del_cambio,
        filas_despues=len(creditos),
    )
)
display(pd.DataFrame(bitacora_limpieza).tail(1))

creditos["ratio_cuota_ingreso"] = creditos["cuota_pesos"] / creditos["ingreso_mensual"]
display(
    creditos["ratio_cuota_ingreso"]
    .describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.9, 0.99])
    .round(3)
    .to_frame()
)
cuota_mayor_que_ingreso = creditos["ratio_cuota_ingreso"] > 1
print(
    f"Créditos con cuota mayor que el ingreso: {cuota_mayor_que_ingreso.sum()} "
    f"({cuota_mayor_que_ingreso.mean():.1%})"
)
print(
    f"Créditos sin ratio (cliente sin ingreso): "
    f"{creditos['ratio_cuota_ingreso'].isna().sum()}"
)
display(
    creditos.groupby("tipo_producto")["ratio_cuota_ingreso"]
    .median()
    .round(3)
    .sort_values()
    .to_frame("ratio_mediano")
)

,tabla,cambio,filas_antes,filas_despues,filas_eliminadas,pct_eliminado
4,créditos,Créditos de clientes que no están en clientes.csv,6051,5955,96,1.59


,ratio_cuota_ingreso
count,5686.000
mean,0.314
std,0.238
min,0.020
1%,0.040
25%,0.141
50%,0.250
75%,0.417
90%,0.626
99%,1.149


Créditos con cuota mayor que el ingreso: 105 (1.8%)
Créditos sin ratio (cliente sin ingreso): 269


,ratio_mediano
tipo_producto,
Adelanto de sueldo,0.141
Préstamo al consumo,0.160
Línea de crédito,0.245
Tarjeta de crédito,0.335
Préstamo automotor,0.449
Crédito hipotecario,0.496


**Qué encontramos:** 96 créditos (1,6%) son de 96 clientes que no están en clientes.csv. La cuota se lleva una mediana del 25% del ingreso del cliente: la mitad de los créditos está entre el 14% y el 42%, y el 90% por debajo del 63%. Hay 105 créditos (1,8%) con una cuota mayor que el ingreso declarado, con un máximo de 1,99 veces.

**Evidencia:** merge de 6.051 créditos con clientes (las mismas 6.051 filas antes y después, y 96 sin datos del cliente). El ratio se calculó para los 5.686 créditos de clientes con ingreso; los otros 269 son de los 156 clientes sin ingreso del punto 1.7. Por tipo de producto el ratio tiene sentido: es más bajo en adelantos de sueldo (14%) y préstamos al consumo (16%) y más alto en automotores (45%) e hipotecarios (50%), que son los créditos más grandes.

**Decisiones:**

- Los 96 créditos sin cliente se sacan (quedan en la bitácora): sin los datos del cliente no se puede calcular el ratio ni el perfil. No sesgan el resultado, porque su mora (36,5%) es parecida a la del total (39,2%).
- Los 105 créditos con cuota mayor que el ingreso se dejan. Son raros pero no imposibles (el ingreso es declarado y el banco no siempre lo verifica), y son justamente el tipo de caso que una política de crédito tiene que poder ver.

**Qué faltaría saber:** el ratio solo mira la cuota de cada crédito. Si el cliente tiene otras deudas, en el banco o en otros bancos, su carga real es mayor.